# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

D. Trevascus  ->  D. Trevascus  |  ['D. Trevascus']
S. Kraus  ->  S. Kraus  |  ['S. Kraus']
A. Pillepich  ->  A. Pillepich  |  ['A. Pillepich']
J. Braspenning  ->  J. Braspenning  |  ['J. Braspenning']
J. Wolf  ->  J. Wolf  |  ['J. Wolf']
S. Belladitta  ->  S. Belladitta  |  ['S. Belladitta']
E. Bañados  ->  E. Bañados  |  ['E. Bañados']
R. E. Hviding  ->  R. E. Hviding  |  ['R. E. Hviding']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
A. d. Graaff  ->  A. D. Graaff  |  ['A. D. Graaff']
M. Schirmer  ->  M. Schirmer  |  ['M. Schirmer']
F. Walter  ->  F. Walter  |  ['F. Walter']
F. Zagaria  ->  F. Zagaria  |  ['F. Zagaria']
M. Benisty  ->  M. Benisty  |  ['M. Benisty']
J. Li  ->  J. Li  |  ['J. Li']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
Y. Wang  ->  Y. Wang  |  ['Y. Wang']
D. Gasman  ->  D. Gasman  |  ['D. Gasman']
M. Samland  ->  M. Samland  |  ['M. Samland']
T. Henning  ->  T. Henning  |  ['T. Henning']
Arxiv has 86 new papers today
          9 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/9 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.31357


extracting tarball to tmp_2611.31357... done.
Retrieving document from  https://arxiv.org/e-print/2611.31363


extracting tarball to tmp_2611.31363...

 done.
Retrieving document from  https://arxiv.org/e-print/2611.31367


extracting tarball to tmp_2611.31367...

... (17 outputs left out)

✔ → 43660:\section{Gas in the disk\label{sec:gas_in_disk}}
  ↳ 61795:\section{The winds and jets}


✔ → 61795:\section{The winds and jets}
  ↳ 120404:\section{Conclusions}
✔ → 120404:\section{Conclusions}
  ↳ 129655:\begin{appendix}
✔ → 129655:\begin{appendix}
  ↳ 129672:\section{Maps of atomic emission in different velocity channels}\label{appendixfigures}


✔ → 129672:\section{Maps of atomic emission in different velocity channels}\label{appendixfigures}
  ↳ 138522:\section{Nonequilibrium Ionization behavior of cooling shocks}\label{NEI}
✔ → 138522:\section{Nonequilibrium Ionization behavior of cooling shocks}\label{NEI}
  ↳ 142531:\section{Overview of the DG Tau star-disk-winds-jet system}
✔ → 142531:\section{Overview of the DG Tau star-disk-winds-jet system}
  ↳ 145947:end


/tmp/ipykernel_2850/2822249172.py:52: LatexWarning: 2611.32856 did not run properly
list index out of range
  warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +


### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.31367-b31b1b.svg)](https://arxiv.org/abs/2611.31367) | **AGN feedback in the multiphase inner intracluster medium of a cosmological super-resolved galaxy cluster**  |
|| M. Prunier, et al. -- incl., <mark>A. Pillepich</mark>, <mark>J. Braspenning</mark> |
|*Appeared on*| *2026-11-25*|
|*Comments*| *Submitted to MNRAS, 19 pages, 13 figures*|
|**Abstract**|            The intracluster medium (ICM) of galaxy cluster cores is a multiphase environment regulated by active galactic nuclei (AGN) feedback. Extended cool gas nebulae with complex morphologies and kinematics are commonly observed in cool-core clusters, but their origin and interaction with AGN feedback remain poorly understood. We present a cosmological magnetohydrodynamical zoom-in simulation of a low-mass cool-core galaxy cluster ($M_{\mathrm{200c}}=2.3\times10^{14}\,\mathrm{M}_\odot$), using initial conditions from TNG-Cluster and the IllustrisTNG galaxy formation model. Through super-Lagrangian refinement, we achieve a baryonic mass resolution of $2.3\times10^{4}\,\mathrm{M}_\odot$ and a spatial resolution of $\sim40\text{-}90$ pc in the cool ($10^{4}\,\mathrm{K}<T<10^{5.5}\,\mathrm{K}$) gas phase within the central $370$ ckpc, while retaining the full cosmological context. We investigate the properties and evolution of cool clouds and filaments and their interaction with AGN feedback at $z\leq0.35$. The cluster core hosts a rich population of dense, low-entropy filaments, including inflowing and outflowing structures and occasional rotationally supported disks. Comparing simulations with AGN kinetic feedback turned off and on, we find that AGN activity fragments and redistributes the cool gas and increases its covering fraction without destroying the cool gas reservoir. Individual outbursts drive transient increases in the velocities and velocity dispersions of both cool and hot gas, on short, $\lesssim 50$ Myr, timescales. We also identify low-entropy gas uplifted by AGN-driven outflows. While feedback globally heats the ICM and offsets cooling, it also creates conditions that may favor local thermal instabilities -- through shock compressions, mixing, turbulence, and uplift.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.31385-b31b1b.svg)](https://arxiv.org/abs/2611.31385) | **Euclid: Discovery of a quasar at z = 8.13 hosting a 360-million-solar-mass black hole**  |
|| <mark>J. Wolf</mark>, et al. -- incl., <mark>S. Belladitta</mark>, <mark>E. Bañados</mark>, <mark>R. E. Hviding</mark>, <mark>K. Jahnke</mark>, <mark>A. d. Graaff</mark>, <mark>M. Schirmer</mark>, <mark>F. Walter</mark> |
|*Appeared on*| *2026-11-25*|
|*Comments*| *Published in Astronomy & Astrophysics, Volume 715, Article A356 on November 24, 2026*|
|**Abstract**|            Quasars powered by rapid accretion onto supermassive black holes (SMBHs) have been uncovered deep into the first gigayear of cosmic history. We present the discovery of EUCL J150452.291+625224.39 (J1504+6252), selected from Euclid Wide Survey imaging and spectroscopically confirmed as the first quasar at z > 8 through observations with Keck, HST, LBT, and JWST. J1504+6252 exhibits the defining spectral signatures of archetypal quasars at lower redshift, including a luminous blue power-law continuum and broad permitted emission lines, indicating a classical broad-line region. JWST/NIRSpec G395M spectroscopy yields a systemic redshift of z = 8.129 +/- 0.005, corresponding to 630 Myr after the Big Bang and shifting the quasar redshift frontier by about 39 Myr (6% of the age of the Universe at that epoch). From H-beta and Mg II, we measured intrinsic broad-line-region FWHM of 3160 +/- 270 and 3740 (+160/-170) km/s, respectively. A widely used single-epoch H-beta calibration gives a reference black hole mass of log10(MBH/Msun) = 8.56 +/- 0.43. Applying a range of calibrations to both Mg II and H-beta yields log10(MBH/Msun) = 8.33-8.89 and Eddington ratios of 0.12-0.44. Assembling a black hole of several hundred million solar masses within the first few hundred million years, despite its currently sub-Eddington accretion rate, requires either a massive seed or sustained, highly efficient early growth. The discovery of a mature, sub-Eddington quasar at z > 8 suggests that substantially more luminous quasars hosting similarly massive black holes may remain to be discovered. Conversely, previously known reionisation-era quasars may represent the rapidly accreting end of a broader population of already massive black holes spanning a wider range of accretion states.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.31408-b31b1b.svg)](https://arxiv.org/abs/2611.31408) | **From residuals to warps: Probing disk geometry with gas kinematics**  |
|| C. N. Kimmig, et al. -- incl., <mark>F. Zagaria</mark>, <mark>M. Benisty</mark> |
|*Appeared on*| *2026-11-25*|
|*Comments*| *accepted to A&A*|
|**Abstract**|            Evidence of shadowing among protoplanetary disks suggests that warped protoplanetary disks are widespread, making it crucial to constrain their three-dimensional shape in order to understand their dynamical evolution and origin. Yet, inferring this warp shape from observations is challenging, as the typical signatures of a warp, especially shadows, are often degenerate with the warp shape. Kinematics of disks may provide a method to recover the shape by interpreting the residuals from a Keplerian disk model as local changes in inclination and position angle. In this work, we test its applicability and limitations. We perform radiative transfer simulations of disks with a parametrized warp shape to create synthetic maps of $^{12}\mathrm{CO}$ rotational line emission at ALMA wavelengths and compare the fitted warp with the true warp shape. We find that the warp shape can be recovered for nearly face-on disks with small warp misalignments of a few degrees, even without knowledge about disk properties such as emission height. The overall warp shape remains traceable even at moderate spectral resolutions up to $0.8\,\mathrm{km/s}$. However, we identify regimes where the method breaks down, including disks with an inclination higher than $40^\circ$, a high emission height of the tracer, and sharp warp transitions. In addition, radial and vertical motions can introduce apparent warps and twists, motivating extensions of the method to account for these effects.         |

## Failed papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.31357-b31b1b.svg)](https://arxiv.org/abs/2611.31357) | **orbitize! v4: Joint Orbit Fitting with Gaia DR4 Astrometry and Flexible Analytic Linear Marginalization**  |
|| C. R. D. Ó, et al. -- incl., <mark>D. Trevascus</mark> |
|*Appeared on*| *2026-11-25*|
|*Comments*| *submitted to JOSS. orbitize! v4 is available here this https URL*|
|**Abstract**|            orbitize! is a Python package for Bayesian orbit fitting of planetary and stellar companions (Blunt et al., 2020). It provides a unified framework for fitting relative astrometry, radial velocities, and absolute astrometry, supporting analyses of both resolved and unresolved systems. Available posterior sampling methods include parallel-tempered Markov chain Monte Carlo (MCMC) through ptemcee (Foreman-Mackey et al., 2013; Vousden et al., 2016) and nested sampling through dynesty (Speagle, 2020), nautilus (Lange, 2023), and pymultinest (Buchner et al., 2014; Feroz et al., 2019). This paper accompanies the release of orbitize! version 4.0 and describes the functionality added since version 3.0 (Blunt et al., 2024), with a particular focus on joint orbit fitting with Gaia DR4 epoch astrometry and the implementation of the Orbits for the Extremely Impatient (OFTeI) algorithm (Blunt et al. 2026b, in preparation).         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: '69117' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.31363-b31b1b.svg)](https://arxiv.org/abs/2611.31363) | **Refining the Architecture of the Quadruple Pre-main-sequence System HD 98800: Improved Orbits, Dynamical Masses, and a Transiting Circumbinary Disk**  |
|| G. Torres, et al. -- incl., <mark>S. Kraus</mark> |
|*Appeared on*| *2026-11-25*|
|*Comments*| *21 pages including figures and tables. Accepted for publication in The Astrophysical Journal*|
|**Abstract**|            We present new spectroscopic and interferometric observations of the pre-main-sequence hierarchical quadruple system HD 98800, a member of the nearby TW Hya association. Our radial velocity monitoring extends the total coverage to over 36 years, sampling the periastron passage in the $\sim$240-yr outer orbit. By combining these velocities with new and archival interferometric observations of both inner binaries (A and B), we significantly refine the system orbits and constrain the dynamical masses of all four stars to $\leq$ 2.6 percent. The resulting stellar properties (masses, effective temperatures, and absolute magnitudes) are found to agree well with current stellar evolution models. Notably, binary B hosts a known circumbinary disk in a polar orientation. Our interferometric imaging, as well as ground-based photometry by others, capture binary A currently passing directly behind the disk, a striking alignment given the outer orbit's long period. We use the photometry to infer the optical extinction produced by the disk material. The eclipse is expected to last about a decade, and offers an unprecedented opportunity to probe the structure and composition of the potentially planet-forming disk. Our refined orbits and component masses provide a robust dynamical framework to support ongoing and future studies of this unique event.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.31419-b31b1b.svg)](https://arxiv.org/abs/2611.31419) | **A New Population of Jovian Exoplanets Suitable For Direct Imaging with the Habitable Worlds Observatory, the Roman Coronagraph, and ELTs Identified from HARPS Radial-Velocity Data and Hipparcos-Gaia Astrometry**  |
|| <mark>J. Li</mark>, et al. |
|*Appeared on*| *2026-11-25*|
|*Comments*| *29 pages, 8 figures, 8 tables; submitted for publication to AJ on Sept 18, 2026; revised to address referee comments and resubmitted. Zenodo link to full set of files for target analysis will be made available in the near future*|
|**Abstract**|            We jointly analyze archival radial-velocity data from the HARPS spectrograph and astrometry from Hipparcos and Gaia to identify and constrain the dynamical properties of new and known long-period jovian exoplanets. Our search yields 14 new confirmed planets with masses of 0.04--6.1 $M_{\rm Jup}$ and semimajor axes of 0.1--12.6 au. Planets include a super-Jupiter in a near-Saturn-like orbit producing a strong astrometric acceleration (HIP 669 b), an eccentric Jupiter around a K star with a modest acceleration (HIP 39039 b), a 74-day-orbit warm Neptune (HIP 64550 b), and a giant planet around a Tier A Habitable Worlds Observatory target that may endanger any habitable zone Earth (HIP 17378 b). Three systems show evidence for candidate planets requiring additional data for confirmation. Our modeling also constrains the dynamical masses and orbits for 32 previously known radial-velocity-discovered planets. Orbits, masses, and ages for our sample show that five targets are potentially detectable in reflected light with the Roman Coronagraph, 28 should be imageable in reflected light with the Habitable Worlds Observatory, and six may be detectable in thermal emission with E-ELT/METIS. These results demonstrate the power of combining archival radial-velocity data with astrometry to discover and characterize long-period giant planets, a capability that will only grow with Gaia DR4. It provides a new sample of targets for future direct-imaging observations, and shows a path forward for timely vetting of targets for exo-Earth imaging with the Habitable Worlds Observatory.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.32656-b31b1b.svg)](https://arxiv.org/abs/2611.32656) | **Constraining the Binary Fraction and Mass-Ratio Distribution of FGK Dwarfs with Gaia XP Spectra: Metallicity, $α$-Element, and Age Dependencies**  |
|| <mark>Y. Zhang</mark>, et al. |
|*Appeared on*| *2026-11-25*|
|*Comments*| *24 pages, 17 figures plus 1 online figure set with 2 additional figures, 5 tables, accepted for publication in ApJ*|
|**Abstract**|            We present a study of the binary fraction and mass-ratio distribution for near 200,000 FGK dwarf stars within 1 kpc, utilizing the Stellar Locus Outlier (SLOT) method applied to synthetic Gaia XP photometry and stellar parameters from LAMOST DR9. By modeling the asymmetric color residuals relative to a metallicity-dependent single-star locus, we measure the binary fraction $f_b$ and constrain the power-law slope $\gamma_{\rm large\,q}$ of the mass-ratio distribution at high mass ratio $q$. Higher metallicity is associated with a steeper (more negative) $\gamma_{{\rm large}q}$, indicating a larger proportion of low-mass-ratio systems. For $\alpha$-elements, [Mg/H] and [Si/H] are anticorrelated with $f_b$ in thin-disk-dominated subsamples; conversely, [Ca/H] shows a positive correlation with $f_b$ in thin-disk-dominated subsamples. No correlation is found between $f_b$ and $\alpha$-element abundances in thick-disk-dominated subsamples. The binary fraction increases modestly toward older ages in the thin-disk population, a trend that likely reflects underlying correlations with chemistry. Our results shed light on how chemical composition and dynamical processes shape the binary properties of solar-type stars.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.32699-b31b1b.svg)](https://arxiv.org/abs/2611.32699) | **The W Transform: an additional powerful tool for astrophysical time-series analysis**  |
|| P. D. S. d. Lima, et al. -- incl., <mark>Y. Wang</mark> |
|*Appeared on*| *2026-11-25*|
|*Comments*| *25 pages, 11 figures, accepted for publication in ApJ*|
|**Abstract**|            This work introduces the W Transform (WT) for astrophysical applications and explores its similarities and differences to the Continuous Wavelet Transform (CWT), in photometric stellar light curves (LCs). We tested both techniques on synthetic data, producing LCs with different noise levels and numbers of gaps, and applied them to LCs from TESS space mission. The analysis focused on six stars: the solar analog TIC 276293046, the ultra-cool dwarf TIC 359892714, the $\gamma$ Dor pulsator TIC 164173105, the eclipsing binary TIC 355151781, the hosting planet star TIC 142276270, and the brown dwarf TIC 102076870. Overall, the results were in agreement, except for stars with LCs presenting sector-to-sector differences in structures or changes in the number of gaps, reflecting time-dynamic effects in the same target, which affect how time-dependent variability features appear in the time-frequency representation. For such LCs, the WT spectra are more stable and accurate in recovering signal amplitudes than the CWT maps, which exhibit distortions and limitations, showing that the inherent flexibility of the WT method in adjusting the dominant frequency could enhance time resolution even in LCs with complex or variable structures. We conclude that the WT is a robust tool for the analysis of stellar LCs, and that both methods may complement each other in an in-depth analysis of a periodic astrophysical signal. The present results suggest that the WT is a robust and accurate tool for analyzing stellar LCs, expanding the range of numerical techniques available for analyzing different stellar periodic phenomena from photometric observations.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.32856-b31b1b.svg)](https://arxiv.org/abs/2611.32856) | **MINDS: The mid-infrared spectrum of the disk, jets, and winds of DG Tau**  |
|| M. Güdel, et al. -- incl., <mark>D. Gasman</mark>, <mark>M. Samland</mark>, <mark>T. Henning</mark> |
|*Appeared on*| *2026-11-25*|
|*Comments*| *30 pages, 28 figures, 2 tables. Accepted (22 October 2026) for publication in Astronomy and Astrophysics*|
|**Abstract**|            DG Tau is one of the best-studied young stellar objects in transition from a Class I protostar to a T Tauri star. It is no longer embedded in a dense molecular cloud, but still reveals features that are commonly seen in protostars at earlier evolutionary stages. DG Tau hosts a protoplanetary disk and two energetic jets that can be studied down to the immediate environment of the star and disk at optical, infrared, and X-ray wavelengths. We use the Medium Resolution Spectrometer of MIRI on the James Webb Space Telescope. DG Tau's mid-infrared spectrum consists of a very bright continuum with various silicate features, and lines of CO, H$_2$O, OH, H I, H$_2$, Ne$^{+}$, Ne$^{2+}$, Ar$^{+}$, Fe$^{+}$, and Ni$^{+}$. We find substantial grain growth to at least 2 $\mu$m at an early evolutionary stage. Lines of H$_2$O and CO are formed at extremely high temperatures (600-2000 K). The ratio of warm to hot water is the lowest among previous observations of disks with JWST. This and a very strong continuum point toward the relevance of the strong accretion of DG Tau and consequently strong stellar UV irradiation. The pair of jets reveals a rich spectrum of emission lines from ions, in particular producing forbidden lines of [Ne II], [Ne III], [Ar II], [Fe II], and [Ni II]. The forward jet shows a collimated structure as found previously, while the hitherto poorly described, receding jet is more widely dispersed on scales of several arcsec, showing a layered structure with the fastest components being the most collimated. The ionization of the jet gas cannot be reconciled with high-energy radiation from the star itself. Instead, we propose a unified nonequilibrium ionization shock model in which the previously studied X-ray jet plasma continuously cools, recombining ions until, at temperatures around $10^4$ K and below, metals at low ionization stages emit the observed ion lines.         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error list index out of range</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.31367/./figures/Kinematic_entropy_uplift_outflowing_gas.png', 'tmp_2611.31367/./figures/Panel_Subbox_Density.png', 'tmp_2611.31367/./figures/Panel_Subbox_vrad.png', 'tmp_2611.31367/./figures/Panel_Pressure.png', 'tmp_2611.31367/./figures/Panel_Density_arrows_.png', 'tmp_2611.31367/./figures/Panel_Beta.png', 'tmp_2611.31367/./figures/Panel_Entropy.png', 'tmp_2611.31367/./figures/Panel_vrad_los_arrows.png', 'tmp_2611.31367/./figures/gas_phase_fractions_sfr_nclouds.png']
copying  tmp_2611.31367/./figures/Kinematic_entropy_uplift_outflowing_gas.png to _build/html/
copying  tmp_2611.31367/./figures/Panel_Subbox_Density.png to _build/html/
copying  tmp_2611.31367/./figures/Panel_Subbox_vrad.png to _build/html/
copying  tmp_2611.31367/./figures/Panel_Pressure.png to _build/html/
copying  tmp_2611.31367/./figures/Panel_Density_arrows_.png to _build/html/
copying  tmp_2611.31367/./figures/Panel_Beta.png to _build/html/
copying  tmp_2611.31367/./figures/Panel_Entropy.png

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\arraystretch}{1.4}$
$\newcommand\rahul{#1}$
$\newcommand\dylan{#1}$
$\newcommand\annalisa{#1}$
$\newcommand\julie{#1}$
$\newcommand\kat{#1}$</div>



<div id="title">

# AGN feedback in the multiphase inner intracluster medium of a cosmological super-resolved galaxy cluster

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.31367-b31b1b.svg)](https://arxiv.org/abs/2611.31367)<mark>Appeared on: 2026-11-25</mark> -  _Submitted to MNRAS, 19 pages, 13 figures_

</div>
<div id="authors">

M. Prunier, et al. -- incl., <mark>A. Pillepich</mark>, <mark>J. Braspenning</mark>

</div>
<div id="abstract">

**Abstract:** The intracluster medium (ICM) of galaxy cluster cores is a multiphase environment regulated by active galactic nuclei (AGN) feedback. Extended cool gas nebulae with complex morphologies and kinematics are commonly observed in cool-core clusters, but their origin and interaction with AGN feedback remain poorly understood.We present a cosmological magnetohydrodynamical zoom-in simulation of a low-mass cool-core galaxy cluster ( $\(M_{\mathrm{200c}}=2.3\times10^{14} \mathrm{M}_\odot\)$ ), using initial conditions from TNG-Cluster and the IllustrisTNG galaxy formation model. Through super-Lagrangian refinement, we achieve a baryonic mass resolution of $\(2.3\times10^{4} \mathrm{M}_\odot\)$ and a spatial resolution of $\(\sim40\text{--}90\)$ pc in the cool ( $\(10^{4} \mathrm{K}<T<10^{5.5} \mathrm{K}\)$ ) gas phase within the central $\(370\)$ ckpc, while retaining the full cosmological context.We investigate the ${properties and evolution}$ of cool clouds and filaments and their interaction with AGN feedback at $z\leq0.35$ . The cluster core hosts a rich population of dense, low-entropy filaments, including inflowing and outflowing structures and occasional rotationally supported disks. Comparing simulations with AGN kinetic feedback turned off and on, we find that AGN activity ${fragments and redistributes}$ the cool gas and increases its covering fraction without destroying the cool gas reservoir. Individual outbursts drive transient increases in the velocities and velocity dispersions of both cool and hot gas, on short, $\lesssim 50$ Myr, timescales. We also identify low-entropy gas uplifted by AGN-driven outflows. While feedback globally heats the ICM and offsets cooling, ${it also creates conditions that may favor local thermal instabilities -- through shock compressions, mixing, turbulence, and uplift.}$

</div>

<div id="div_fig1">

<img src="tmp_2611.31367/./figures/Kinematic_entropy_uplift_outflowing_gas.png" alt="Fig13.1" width="33%"/><img src="tmp_2611.31367/./figures/Panel_Subbox_Density.png" alt="Fig13.2" width="33%"/><img src="tmp_2611.31367/./figures/Panel_Subbox_vrad.png" alt="Fig13.3" width="33%"/>

**Figure 13. -** Uplift of low-entropy gas. **Top panel:** Mass density profiles of the outflowing low-entropy gas between $-20$ Myr and $70$ Myr relative to the feedback event (red dashed line). The inset shows the corresponding total mass of low-entropy gas within 80 kpc across time. Following the outburst, there is an increase in the mass density of outflowing low-entropy gas between $20$ and $70$ kpc, as well as a global increase in its total mass, consistent with this gas being uplifted or pushed by the outburst. **Bottom images:** Gas density and radial velocity maps (170 $\times$ 170 kpc) of the low-entropy ($<50 \mathrm{keV cm^2}$) gas at representative chosen times before and after an SMBH directional kinetic feedback event at $z=0.238$(E = $3.8 \times 10^{58}$ erg). The directional kinetic energy injection produces an underdense cavity with an upward-moving shock front that partially disrupts the dense disk-like structure and entrains trailing dense gas.  (*fig:res_entropy_upflift*)

</div>
<div id="div_fig2">

<img src="tmp_2611.31367/./figures/Panel_Pressure.png" alt="Fig8.1" width="20%"/><img src="tmp_2611.31367/./figures/Panel_Density_arrows_.png" alt="Fig8.2" width="20%"/><img src="tmp_2611.31367/./figures/Panel_Beta.png" alt="Fig8.3" width="20%"/><img src="tmp_2611.31367/./figures/Panel_Entropy.png" alt="Fig8.4" width="20%"/><img src="tmp_2611.31367/./figures/Panel_vrad_los_arrows.png" alt="Fig8.5" width="20%"/>

**Figure 8. -** The complex configurations and kinematics of the multiphase gas in h963. We show 200 by 200 kpc maps (+/- 20 kpc depth) at four different redshifts, with rows displaying gas pressure, density, {inverse plasma beta (magnetic pressure over thermal gas pressure)}, entropy of the low-entropy gas \citep[$K < 50 \mathrm{keV cm^2}$;][]{2008Cavagnolo,2009Cavagnolo}, and mass-weighted gas radial velocity of the low-entropy gas (with respect to the central SMBH, velocity color scale is capped to better highlight the inflowing and outflowing gas). The black crosses denote the position of the central most massive SMBH. {The white arrows at $z=0.33$ and $z=0.08$ point to high-velocity outflowing gas in the path of AGN feedback outbursts.} The cool, low-entropy, and higher-density gas can take a variety of configurations, from galactic disks, extended filaments, and clumpy structures to small cloudlets. These interact with AGN feedback-driven outflows, cavities, and shock fronts, visible in the density and pressure maps. Filaments are mainly inflowing, with radial velocities spanning $ -1000 \text{to} 600 \mathrm{km s^{-1}}$. (*fig:res_panel*)

</div>
<div id="div_fig3">

<img src="tmp_2611.31367/./figures/gas_phase_fractions_sfr_nclouds.png" alt="Fig2" width="100%"/>

**Figure 2. -** {Gas phases with AGN kinetic feedback on or off. We show the gas mass fractions within 200 ckpc of the BCG center for cold, warm, and star-forming gas as a function of time, for the runs with AGN kinetic feedback on (blue) and off (light red).  We focus on this central region to highlight the impact of AGN feedback. The left $y$-axis shows the median number of cold clouds over the redshift range considered. With AGN feedback active, the number of cold clouds increases from $\sim200$ to $\sim450$ despite similar cold- and warm-gas mass fractions, while the feedback-off run develops a larger reservoir of star-forming gas, indicating more efficient conversion of cold gas into stars.} (*fig:glob_quant_with_without*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.31367"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\feii}{\ion{Fe}{II}\xspace}$
$\newcommand{\oi}{\ionF{O}{I}{~\lambda6300}\xspace}$
$\newcommand{\ciii}{\ion{C}{III}]~\lambda1909\xspace}$
$\newcommand{\siiii}{\ion{Si}{III}]~\lambda1892\xspace}$
$\newcommand{\heii}{\ion{He}{II}~\lambda4686\xspace}$
$\newcommand{\nv}{\ion{N}{V}~\lambda1240\xspace}$
$\newcommand{\ovi}{\ion{O}{VI}~\lambda1034\xspace}$
$\newcommand{\mgii}{\ion{Mg}{II}~\lambda2800\xspace}$
$\newcommand{\mgiishort}{\ion{Mg}{II}\xspace}$
$\newcommand{\oiii}{\ionF{O}{III}{~\lambda5007}\xspace}$
$\newcommand{\oii}{\ionF{O}{II}{~\lambda3727}\xspace}$
$\newcommand{\oiiil}{\ionF{O}{III}{~\lambda4959}\xspace}$
$\newcommand{\oiiidd}{\ionF{O}{III}{~\lambda\lambda4959,5007}\xspace}$
$\newcommand{\oiiishort}{\ionF{O}{III} \xspace}$
$\newcommand{\hbeta}{H \beta~\lambda4861\xspace}$
$\newcommand{\hbetashort}{H \beta\xspace}$
$\newcommand{\hgammafull}{H \gamma~\lambda4340\xspace}$
$\newcommand{\hgamma}{H \gamma\xspace}$
$\newcommand{\civ}{\ion{C}{IV}~\lambda1549\xspace}$
$\newcommand{\civshort}{\ion{C}{IV}\xspace}$
$\newcommand{\lyafull}{Ly \alpha~\lambda1216\xspace}$
$\newcommand{\lya}{Ly \alpha\xspace}$
$\newcommand{\nvdd}{\ion{N}{V}~\lambda\lambda1238,1242\xspace}$
$\newcommand{\lyanv}{Ly \alpha+ \ion{N}{V}~complex\xspace}$
$\newcommand{\paa}{Pa \alpha~\lambda18751\xspace}$
$\newcommand{\pab}{Pa \beta~\lambda12818\xspace}$
$\newcommand{\pag}{Pa \gamma~\lambda10938\xspace}$
$\newcommand{\pad}{Pa \delta~\lambda10049\xspace}$
$\newcommand{\nii}{\ionF{N}{II}{~\lambda\lambda6548,6583}\xspace}$
$\newcommand{\sii}{\ionF{S}{II}{~\lambda\lambda6716,6731}\xspace}$
$\newcommand{\cii}{\ionF{C}{II}{~\lambda157.6 \mum}\xspace}$
$\newcommand{\siv}{\ion{S}{IV}~\lambda\lambda1394,1403\xspace}$
$\newcommand{\halpha}{H \alpha~\lambda6563\xspace}$
$\newcommand{\oiiiaur}{\ionF{O}{III}{~\lambda4363}\xspace}$
$\newcommand{\orcid}[1]\begin$
$\newcommand{\arraystretch}{1.16}$
$\newcommand{\arraystretch}{1.10}$</div>



<div id="title">

# _Euclid_: Discovery of a quasar at $z=8.13$ hosting a 360-million-solar-mass black hole $\thanks{This paper is published on behalf of the Euclid Consortium.}$

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.31385-b31b1b.svg)](https://arxiv.org/abs/2611.31385)<mark>Appeared on: 2026-11-25</mark> -  _Published in Astronomy & Astrophysics, Volume 715, Article A356 on November 24, 2026_

</div>
<div id="authors">

<mark>J. Wolf</mark>, et al. -- incl., <mark>S. Belladitta</mark>, <mark>E. Bañados</mark>, <mark>R. E. Hviding</mark>, <mark>K. Jahnke</mark>, <mark>A. d. Graaff</mark>, <mark>M. Schirmer</mark>, <mark>F. Walter</mark>

</div>
<div id="abstract">

**Abstract:** Quasars powered by rapid accretion onto supermassive black holes (SMBHs) have been uncovered deep into the first gigayear of cosmic history. Here we present the discovery of EUCL J150452.291+625224.39 (J1504+6252), selected from Euclid Wide Survey imaging and spectroscopically confirmed as the first quasar at $z>8$ through a campaign with the Keck Observatory, the _Hubble_ Space Telescope, the Large Binocular Telescope, and the _James Webb_ Space Telescope (JWST). J1504+6252 exhibits the defining spectral signatures of archetypal quasars observed at a lower redshift, including a luminous blue power-law continuum and broad permitted emission lines, indicating the presence of a classical broad-line region. Using medium-resolution JWST/NIRSpec G395M spectroscopy, we determined a systemic redshift of $z_{\rm sys}=8.129\pm0.005$ , corresponding to only $630 \mathrm{Myr}$ after the Big Bang and shifting the quasar redshift frontier by $\simeq39 \mathrm{Myr}$ , or approximately $6\%$ of the age of the Universe at that epoch.From the $\hbetashort$ and $\mgiishort$ emission lines, we measured intrinsic broad-line-region full widths at half maximum  FWHM( $\hbetashort$ ,BLR)= $3160^{+270}_{-270} \mathrm{km s^{-1}}$ and FWHM( $\mgiishort$ ,BLR) $=3740^{+160}_{-170} \mathrm{km s^{-1}}$ , respectively. From a widely used single-epoch $\hbetashort$ -based calibration, we measured a reference black hole mass of $\logten(M_{\rm BH}/M_\odot)=8.56^{+0.43}_{-0.43}$ . Applying a range of single-epoch black hole mass calibrations to both $\mgiishort$ and $\hbetashort$ yields a broader interval of $\logten(M_{\rm BH}/M_\odot)=8.33$ -- $8.89$ , with corresponding Eddington ratios of $\lambda_{\rm Edd}=0.12$ -- $0.44$ . Assembling a black hole of several hundred million solar masses within the first few hundred million years, despite its currently sub-Eddington accretion rate, requires either a massive seed or sustained, highly efficient early growth. The discovery of a mature, sub-Eddington quasar at $z>8$ suggests that substantially more luminous quasars hosting similarly massive black holes may remain to be discovered. Conversely, the previously known re-ionisation-era quasars may represent the rapidly accreting end of a broader population of already massive black holes spanning a wider range of accretion states.

</div>

<div id="div_fig1">

<img src="tmp_2611.31385/./J1504_spectroscopic_confirmation_fluxcalibrated_noJWST.png" alt="Fig5" width="100%"/>

**Figure 5. -** Combined spectroscopic confirmation of J1504+6252. The upper panels show the two-dimensional spectra from HST/WFC3 G141 and LBT/LUCI G200 zJspec and HKspec. The lower panel compares the extracted HST/WFC3 G141 spectrum with the ground-based LUCI spectra and the best-fitting quasar template, with the spectra placed on a common flux scale using the \Euclid\YE, \JE, and \HE photometry. The \Euclid photometric measurements are shown as open symbols, and the shaded bands indicate the corresponding \Euclid/NISP filter transmission curves \citep[][]{Schirmer-EP18}. The sharp continuum break at $\lambda_{\rm obs}\simeq1.1 \micron$ is identified as the Ly$ \alpha$ break, yielding a template redshift of $z_{\rm fit}=8.09$ with $\chi^2_\nu=0.86$.
    Grey shading shows the $1\sigma$ uncertainty on the HST/WFC3 spectrum. Vertical dashed lines mark the expected observed wavelengths of prominent rest-frame UV emission lines at the best-fitting redshift. (*fig:spec*)

</div>
<div id="div_fig2">

<img src="tmp_2611.31385/./qso_combined_Hbeta_MgII_fit.png" alt="Fig2" width="100%"/>

**Figure 2. -** 
Fit to the $\hbeta$short--$\oi$iishort(top) and $\mgii$short(bottom) regions of J1504+6252. For each region, the upper panel shows the JWST/NIRSpec spectral fit.
The observed spectrum is shown as the dark grey histogram, and the $1\sigma$ flux uncertainty is indicated in light grey. The best-fitting total model is shown as the solid black curve, while the continuum is shown as the dashed magenta curve. Individual emission-line model components are shown as coloured offset curves. In the $\hbeta$short panel, the shaded purple region marks the rest-frame wavelength interval used to measure $R_{\rm FeII}$ from the $\feii$ template component. In the $\mgii$short panel, the hatched orange region marks the automatically masked narrow absorption region excluded from the fit. Vertical dashed lines mark the expected observed wavelengths of systemic narrow $\hbeta$short, $\oi$iishort, and $\mgii$short. No significant $\oi$iishort emission is detected. The lower panels show the residuals in units of the flux uncertainty. The horizontal lines in the residual panel delimit the 3$\sigma$ range. (*fig:fit_hb*)

</div>
<div id="div_fig3">

<img src="tmp_2611.31385/./MBH_vs_Lbol_target_mass_range.png" alt="Fig3" width="100%"/>

**Figure 3. -** $L_{\rm bol}$ versus $M_{\rm BH}$ for quasars $z>6.0$ from the literature \citep[][and references therein]{belladitta2025} and J1504+6252. Brown symbols mark the previously most distant quasars with black hole mass measurements \citep{banados2018a,Yang2020,wang2021}. The two stars show the $\hbeta$short and $\mgii$short based  $M_\mathrm{BH}$ measurements for  J1504+6252, using the calibrations from \citet{Vestergaard2006} and \citet{vestergaardosmer2009}, respectively. Dashed lines indicate constant Eddington ratios. J1504+6252 is offset in terms of its Eddington rate with respect to the bulk of the known $z>6$ quasar population. (*fig:mbhlbol*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.31385"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\todo}[1]{\textcolor{cyan}{\bf Todo: #1}}$</div>



<div id="title">

# From residuals to warps: Probing disk geometry with gas kinematics\    

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.31408-b31b1b.svg)](https://arxiv.org/abs/2611.31408)<mark>Appeared on: 2026-11-25</mark> -  _accepted to A&A_

</div>
<div id="authors">

C. N. Kimmig, et al. -- incl., <mark>F. Zagaria</mark>, <mark>M. Benisty</mark>

</div>
<div id="abstract">

**Abstract:** Evidence of shadowing among protoplanetary disks suggests that warped protoplanetary disks are widespread, making it crucial to constrain their three-dimensional shape in order to understand their dynamical evolution and origin. Yet, inferring this warp shape from observations is challenging, as the typical signatures of a warp, especially shadows, are often degenerate with the warp shape. Kinematics of disks may provide a method to recover the shape by interpreting the residuals from a Keplerian disk model as local changes in inclination and position angle. In this work, we test its applicability and limitations. We perform radiative transfer simulations of disks with a parametrized warp shape to create synthetic maps of $^{12}\mathrm{CO}$ rotational line emission at ALMA wavelengths and compare the fitted warp with the true warp shape.We find that the warp shape can be recovered for nearly face-on disks with small warp misalignments of a few degrees, even without knowledge about disk properties such as emission height. The overall warp shape remains traceable even at moderate spectral resolutions up to $0.8 \mathrm{km/s}$ .However, we identify regimes where the method breaks down, including disks with an inclination higher than $40^\circ$ , a high emission height of the tracer, and sharp warp transitions. In addition, radial and vertical motions can introduce apparent warps and twists, motivating extensions of the method to account for these effects.

</div>

<div id="div_fig1">

<img src="tmp_2611.31408/./fitted-curves-10deg.png" alt="Fig2" width="100%"/>

**Figure 2. -** Warp tilt profiles of the simulation with $\beta_\mathrm{0}=5^\circ$, with the residuals calculated using the best case (left panel) and worst case model (right panel). Gray dashed lines show the model input, gray solid lines the best fit from \texttt{warpfitter} and the light grey lines show the posterior distribution from the Gausian process fit by \texttt{warpfitter}. The orange and blue dashed lines indicate the fit according to Eq. \ref{eq:incl-ini}(allowing for a constant offset $c$) of best and worst case model, respectively. The gray shaded region shows regions we exclude from the fit. (*fig:fits-10deg*)

</div>
<div id="div_fig2">

<img src="tmp_2611.31408/./spectral-resolution-incl30.png" alt="Fig19" width="100%"/>

**Figure 19. -** Warp profiles for a twisted disk with $\beta_\mathrm{0}=5^\circ$ and $\gamma_\mathrm{0}=540^\circ$, where the outer disk is inclined by $30^\circ$ for different spectral resolutions. The top panel shows the profiles in the best case scenario, while the bottom panel shows the worst case scenario. (*fig:spectralres-incl30*)

</div>
<div id="div_fig3">

<img src="tmp_2611.31408/./hydro-sloshing.png" alt="Fig26" width="100%"/>

**Figure 26. -** Fitted warp profile $\beta(r)$ , twist $\gamma(r)$, and warp amplitude $\psi(r)$ for different times in a hydrodynamic simulation of a warped, but untwisted disk. We define $r_0=5.2 \mathrm{au}$. The solid lines are obtained with \texttt{warpfitter} at an outer disk inclination of $i_0=10^\circ$. The dotted lines show the \texttt{warpfitter} fit of the opposite side of the disk (flipped case).
    The dashed lines show the true profiles extracted from the 3D hydrodynamic simulations.
    The gray shaded regions indicate where the true $\beta$ is close to zero, hence the twist $\gamma$ is undefined. (*fig:hydro-simulation*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.31408"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

163  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

19  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

7  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
